# Human-Human Agreement Benchmark (paper Section 5.5)

Three additional examiners (AE1–AE3) graded the 510-response held-out test set with the same rubric,
without access to the model grades or to each other's grades. The original examiner labels
(from the dataset) are the reference.

**Sections:**
1. Load Grades
2. Pairwise Human-Human Agreement (paper Table 11)
3. Multi-Rater Reliability (paper Table 12)
4. Fine-Tuned Model vs the Human-Human Range

In [1]:
from itertools import combinations

import numpy as np
import pandas as pd

RUBRIC_COLUMNS = ['Clarity_Score', 'Terminology_Score', 'Coverage_Score', 'Accuracy_Score']
RUBRICS = ['Clarity', 'Terminology', 'Coverage', 'Accuracy']
N_BOOTSTRAP = 10000
SEED = 42

## 1. Load Grades

In [2]:
# Original reference labels = the human scores stored with the fine-tuned test results
ft = pd.read_csv('../Finetuning/finetuned_results.csv')
grades = pd.DataFrame({'ID': ft['student_id'],
                       'Original': sum(ft[f'Human_{r}'] for r in RUBRICS) / 2})

for examiner in ['AE1', 'AE2', 'AE3']:
    ae = pd.read_excel(f'../Human Benchmark/Test_Set_{examiner}.xlsx', sheet_name='Test Set')
    assert not ae['ID'].duplicated().any()
    ae_grade = pd.DataFrame({'ID': ae['ID'].astype(str).str.strip(),
                             examiner: ae[RUBRIC_COLUMNS].apply(pd.to_numeric).sum(axis=1) / 2})
    grades = grades.merge(ae_grade, on='ID', how='inner')

scores = grades[['Original', 'AE1', 'AE2', 'AE3']]
print(f'Shared responses: {len(scores)}')
scores.describe().round(3)

Shared responses: 510


,Original,AE1,AE2,AE3
count,510.000,510.000,510.000,510.000
mean,2.774,3.008,2.722,2.856
std,1.759,1.842,1.885,1.871
min,0.000,0.000,0.000,0.000
25%,1.000,1.000,0.500,1.000
50%,3.000,3.500,3.000,3.500
75%,4.500,5.000,4.500,4.500
max,5.000,5.000,5.000,5.000


## 2. Pairwise Human-Human Agreement

In [3]:
def quadratic_weighted_kappa(a, b):
    """QWK on the 0-5 final grade (0.5 steps), using all grade levels observed in either rater."""
    labels = np.array(sorted(set(a.tolist()) | set(b.tolist())))
    index = {label: i for i, label in enumerate(labels)}
    observed = np.zeros((len(labels), len(labels)))
    for x, y in zip(a, b):
        observed[index[x], index[y]] += 1
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / observed.sum()
    positions = np.arange(len(labels))
    weights = (positions[:, None] - positions[None, :]) ** 2 / (len(labels) - 1) ** 2
    return 1.0 - np.sum(weights * observed) / np.sum(weights * expected)


rows = []
for left, right in combinations(scores.columns, 2):
    a, b = scores[left].to_numpy(), scores[right].to_numpy()
    diff = b - a
    rows.append({'Comparison': f'{left} vs {right}',
                 'QWK': quadratic_weighted_kappa(a, b),
                 'MAE': np.mean(np.abs(diff)),
                 '±1 Acc (%)': np.mean(np.abs(diff) <= 1.0) * 100,
                 'Bias': np.mean(diff)})

pairwise = pd.DataFrame(rows).sort_values('QWK', ascending=False).reset_index(drop=True)
print('Pairwise human-human agreement (510 test responses):')
display(pairwise.round(3))
print(f"Range: QWK {pairwise['QWK'].min():.3f}–{pairwise['QWK'].max():.3f}, "
      f"MAE {pairwise['MAE'].min():.3f}–{pairwise['MAE'].max():.3f}, "
      f"±1 Acc {pairwise['±1 Acc (%)'].min():.2f}–{pairwise['±1 Acc (%)'].max():.2f}%")

Pairwise human-human agreement (510 test responses):


,Comparison,QWK,MAE,±1 Acc (%),Bias
0,AE2 vs AE3,0.903,0.525,88.627,0.134
1,Original vs AE2,0.852,0.660,80.588,-0.052
2,Original vs AE3,0.819,0.745,76.078,0.082
3,Original vs AE1,0.788,0.834,75.882,0.234
4,AE1 vs AE3,0.782,0.805,76.078,-0.152
5,AE1 vs AE2,0.780,0.802,75.098,-0.286


Range: QWK 0.780–0.903, MAE 0.525–0.834, ±1 Acc 75.10–88.63%


## 3. Multi-Rater Reliability

ICC uses a two-way random, absolute-agreement model. CIs: percentile bootstrap over responses (B = 10,000, seed = 42).

In [4]:
def anova_components(values):
    n, k = values.shape
    row_means, col_means, grand = values.mean(axis=1), values.mean(axis=0), values.mean()
    ms_rows = k * np.sum((row_means - grand) ** 2) / (n - 1)
    ms_cols = n * np.sum((col_means - grand) ** 2) / (k - 1)
    residual = values - row_means[:, None] - col_means[None, :] + grand
    ms_error = np.sum(residual ** 2) / ((n - 1) * (k - 1))
    return ms_rows, ms_cols, ms_error


def icc(values):
    n, k = values.shape
    ms_rows, ms_cols, ms_error = anova_components(values)
    icc_2_1 = (ms_rows - ms_error) / (ms_rows + (k - 1) * ms_error + k * (ms_cols - ms_error) / n)
    icc_2_k = (ms_rows - ms_error) / (ms_rows + (ms_cols - ms_error) / n)
    return icc_2_1, icc_2_k


def krippendorff_alpha_interval(values):
    n_valid = values.shape[1]
    observed = np.sum(2.0 * n_valid * np.sum((values - values.mean(axis=1, keepdims=True)) ** 2, axis=1)) \
        / (len(values) * n_valid * (n_valid - 1))
    all_values = values.ravel()
    expected = 2.0 * np.sum((all_values - all_values.mean()) ** 2) / (len(all_values) - 1)
    return 1.0 - observed / expected


def cronbach_alpha(values):
    k = values.shape[1]
    return k / (k - 1) * (1.0 - values.var(axis=0, ddof=1).sum() / values.sum(axis=1).var(ddof=1))


def all_measures(values):
    icc_2_1, icc_2_k = icc(values)
    return [icc_2_1, icc_2_k, krippendorff_alpha_interval(values), cronbach_alpha(values)]


values = scores.to_numpy(dtype=float)
point = all_measures(values)

rng = np.random.default_rng(SEED)
boot = np.array([all_measures(values[rng.integers(0, len(values), len(values))]) for _ in range(N_BOOTSTRAP)])
low, high = np.percentile(boot, [2.5, 97.5], axis=0)

reliability = pd.DataFrame({
    'Measure': ['ICC(2,1), single label set', f'ICC(2,{values.shape[1]}), mean of {values.shape[1]} label sets',
                "Krippendorff's alpha (interval)", "Cronbach's alpha"],
    'Estimate': point, 'CI low': low, 'CI high': high,
})
print('Multi-rater reliability (Original + AE1-AE3):')
reliability.round(3)

Multi-rater reliability (Original + AE1-AE3):


,Measure,Estimate,CI low,CI high
0,"ICC(2,1), single label set",0.821,0.793,0.846
1,"ICC(2,4), mean of 4 label sets",0.948,0.939,0.957
2,Krippendorff's alpha (interval),0.821,0.792,0.846
3,Cronbach's alpha,0.950,0.940,0.958


## 4. Fine-Tuned Model vs the Human-Human Range

In [5]:
model_grade = ft['LLM_Final'].to_numpy()
original = ft['Human_Final'].to_numpy()
model_row = {'QWK': quadratic_weighted_kappa(original, model_grade),
             'MAE': np.mean(np.abs(model_grade - original)),
             '±1 Acc (%)': np.mean(np.abs(model_grade - original) <= 1.0) * 100}

for metric, value in model_row.items():
    lo, hi = pairwise[metric].min(), pairwise[metric].max()
    print(f'{metric:11s} model vs Original = {value:.3f}   human-human range = {lo:.3f}–{hi:.3f}   '
          f'within range: {lo <= value <= hi}')

QWK         model vs Original = 0.821   human-human range = 0.780–0.903   within range: True
MAE         model vs Original = 0.713   human-human range = 0.525–0.834   within range: True
±1 Acc (%)  model vs Original = 78.431   human-human range = 75.098–88.627   within range: True
